# Problema das 8 Rainhas com Programação Genética

## Representação do indivíduo
Cada indivíduo representa uma possível solução do problema das 8 rainhas e é codificado como um vetor de tamanho 8:

- Índice → linha do tabuleiro
- Valor → coluna onde a rainha é colocada

Exemplo:
[0, 4, 7, 5, 2, 6, 1, 3]

Isso garante exatamente uma rainha por linha.

---

## População inicial
A população inicial é formada por indivíduos aleatórios, com valores inteiros entre 0 e 7.
Essa diversidade inicial é essencial para a exploração do espaço de busca.

---

## Função de fitness
A função de fitness mede a qualidade de uma solução contando quantos pares de rainhas **não se atacam**.

- Existem 28 pares possíveis no tabuleiro 8×8
- Fitness máximo = 28 (solução perfeita)

---

## Seleção
A seleção é feita por torneio, onde indivíduos mais aptos têm maior chance de reprodução, mas indivíduos menos aptos ainda podem ser escolhidos, preservando diversidade.

---

## Crossover com número variável de filhos
No processo de crossover:
- Dois pais são selecionados
- O número de filhos é escolhido aleatoriamente entre **0 e 9**
- Cada filho herda genes misturados dos pais
- Cada filho é um novo indivíduo geneticamente distinto

Esse mecanismo aumenta significativamente a variabilidade genética.

---

## Mutação
A mutação é aplicada gene a gene com:

- Probabilidade fixa de **1%**
- Caso ocorra, a coluna da rainha é alterada aleatoriamente

A mutação evita convergência prematura e introduz novas estruturas genéticas.

---

## Archive de soluções
Sempre que um indivíduo atinge fitness máximo (28), ele é salvo em um archive:
- O archive não permite duplicatas
- O objetivo é encontrar **todas as 92 soluções distintas**

---

## Critério de parada
O algoritmo termina quando:
- Todas as 92 soluções forem encontradas
  **OU**
- Um número máximo de gerações for atingido

In [1]:
# ============================================================
# PROBLEMA DAS 8 RAINHAS COM PROGRAMAÇÃO GENÉTICA
# ============================================================
# - Crossover com 0 a 15 filhos
# - Mutação de 1%
# - População dinâmica
# - Archive com todas as soluções
# - Log a cada geração
# ============================================================

import random
import copy
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from IPython.display import HTML
from deap import base, creator, tools


# -----------------------------
# PARÂMETROS GERAIS
# -----------------------------
N = 8
FITNESS_MAX = 28
TARGET_SOLUCOES = 92

POP_INICIAL = 300
MIN_POP = 200
MAX_POP = 800

MAX_GERACOES = 5000
MUT_P = 0.01           # 1% de mutação
MAX_FILHOS = 9         # crossover variável

# -----------------------------
# FUNÇÕES DO PROBLEMA
# -----------------------------
def criar_individuo():
    """Cria um indivíduo aleatório (uma rainha por linha)."""
    return [random.randint(0, N-1) for _ in range(N)]
 
def contar_conflitos(board):
    """Conta conflitos entre rainhas."""
    conflitos = 0
    for i in range(N):
        for j in range(i + 1, N):
            if board[i] == board[j]:
                conflitos += 1
            elif abs(board[i] - board[j]) == abs(i - j):
                conflitos += 1
    return conflitos

def avaliar(ind):
    """Fitness = pares NÃO atacantes."""
    return FITNESS_MAX - contar_conflitos(ind),

def mutacao(ind):
    """Mutação ponto a ponto com taxa de 1%."""
    for i in range(N):
        if random.random() < MUT_P:
            ind[i] = random.randint(0, 7)
    return ind,


def escolher_num_filhos():
    """
    Escolhe o número de filhos com probabilidade enviesada:
    - 1,2,3 são mais prováveis
    - 0,4,5 menos prováveis
    - 6 a 9 cada vez menos provável
    """
    valores = list(range(10))

    pesos = [
        10,   # 0
        30,   # 1
        25,   # 2
        15,   # 3
        8.5,  # 4
        5,    # 5
        3,    # 6
        2,    # 7
        1.,   # 8
        0.5,  # 9
 
    ]

    return random.choices(valores, weights=pesos, k=1)[0]

def crossover_variavel(pai1, pai2):
    """
    Gera aleatoriamente entre 0 e 15 filhos.
    Cada filho herda genes aleatórios de cada pai.
    """
    filhos = []
    qtd = escolher_num_filhos()

    for _ in range(qtd):
        filho = [random.choice([pai1[i], pai2[i]]) for i in range(N)]
        filhos.append(filho)

    return filhos

# -----------------------------
# CONFIGURAÇÃO DEAP
# -----------------------------
if not hasattr(creator, "FitnessMax"):
    creator.create("FitnessMax", base.Fitness, weights=(1.0,))
if not hasattr(creator, "Individuo"):
    creator.create("Individuo", list, fitness=creator.FitnessMax)

toolbox = base.Toolbox()
toolbox.register("individual", tools.initIterate, creator.Individuo, criar_individuo)
toolbox.register("population", tools.initRepeat, list, toolbox.individual)
toolbox.register("evaluate", avaliar)
toolbox.register("select", tools.selTournament, tournsize=3)
toolbox.register("mutate", mutacao)
toolbox.register("clone", copy.deepcopy)

# -----------------------------
# ALGORITMO GENÉTICO
# -----------------------------
def resolver_8_rainhas_GP():
    pop_size = POP_INICIAL
    populacao = toolbox.population(pop_size)
    archive = set() 
    historico_melhores = []

    for geracao in range(1, MAX_GERACOES + 1):

        fitness_vals = []

        # Avaliação
        for ind in populacao:
            ind.fitness.values = toolbox.evaluate(ind)
            fit = ind.fitness.values[0]
            fitness_vals.append(fit)

            if fit == FITNESS_MAX:
                archive.add(tuple(ind))

        best = max(fitness_vals)
        avg = sum(fitness_vals) / len(fitness_vals)

        # Guarda melhor para animação
        melhor_ind = max(populacao, key=lambda ind: ind.fitness.values[0])
        historico_melhores.append(list(melhor_ind))

        # LOG DA GERAÇÃO
        print(
            f"[Geração {geracao:04d}] "
            f"Pop={len(populacao)} | "
            f"Best={best} | "
            f"Média={avg:.2f} | "
            f"Soluções={len(archive)}/{TARGET_SOLUCOES}"
        )

        # Parada
        if len(archive) >= TARGET_SOLUCOES:
            print("✅ Todas as soluções encontradas!")
            break

        # Seleção
        pais = toolbox.select(populacao, len(populacao))
        nova_pop = []

        # Crossover variável
        for i in range(0, len(pais) - 1, 2):
            filhos = crossover_variavel(pais[i], pais[i+1])
            for f in filhos:
                f = creator.Individuo(f)
                toolbox.mutate(f)
                nova_pop.append(f)

        # Controle de população dinâmica
        if nova_pop:
            populacao = nova_pop
        else:
            populacao = toolbox.population(pop_size)

        # Reinicializa população após solução perfeita
        if best == FITNESS_MAX:
            pop_size = min(pop_size + 50, MAX_POP)
            populacao = toolbox.population(pop_size)

        if geracao % 50 == 0:
            pop_size = max(pop_size - 50, MIN_POP)

    return archive, historico_melhores

# -----------------------------
# EXECUÇÃO
# -----------------------------
archive, historico = resolver_8_rainhas_GP()
print(f"\nTotal de soluções únicas encontradas: {len(archive)}")




[Geração 0001] Pop=300 | Best=25.0 | Média=19.92 | Soluções=0/92
[Geração 0002] Pop=307 | Best=26.0 | Média=21.00 | Soluções=0/92
[Geração 0003] Pop=359 | Best=26.0 | Média=21.42 | Soluções=0/92
[Geração 0004] Pop=383 | Best=26.0 | Média=21.55 | Soluções=0/92
[Geração 0005] Pop=412 | Best=27.0 | Média=21.96 | Soluções=0/92
[Geração 0006] Pop=479 | Best=27.0 | Média=22.27 | Soluções=0/92
[Geração 0007] Pop=560 | Best=27.0 | Média=22.42 | Soluções=0/92
[Geração 0008] Pop=596 | Best=27.0 | Média=22.46 | Soluções=0/92
[Geração 0009] Pop=626 | Best=27.0 | Média=22.75 | Soluções=0/92
[Geração 0010] Pop=721 | Best=27.0 | Média=22.94 | Soluções=0/92
[Geração 0011] Pop=877 | Best=28.0 | Média=23.16 | Soluções=1/92
[Geração 0012] Pop=350 | Best=25.0 | Média=20.20 | Soluções=1/92
[Geração 0013] Pop=410 | Best=25.0 | Média=21.02 | Soluções=1/92
[Geração 0014] Pop=435 | Best=27.0 | Média=21.24 | Soluções=1/92
[Geração 0015] Pop=510 | Best=26.0 | Média=21.62 | Soluções=1/92
[Geração 0016] Pop=545 | 